[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import torch.nn as nn
import math

In [13]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.num_heads = num_heads
        self.d_model = d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )


    def forward(self, x):
      B, S, D = x.shape
      x1 = self.ln1(x)

      d_head = self.d_model // self.num_heads
      Q = self.W_q(x1).reshape(B, S, self.num_heads, d_head).transpose(1,2)
      K = self.W_k(x1).reshape(B, S, self.num_heads, d_head).transpose(1,2)
      V = self.W_v(x1).reshape(B, S, self.num_heads, d_head).transpose(1,2)

      score = Q @ K.transpose(-1, -2) / math.sqrt(d_head)

      mask = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
      score.masked_fill_(mask, float('-inf'))
      score = torch.softmax(score, dim=-1)

      out = score @ V
      out = out.transpose(1,2).reshape(B, S, D)
      out = self.W_o(out)

      attn_out = out + x

      mlp_out = self.mlp(self.ln2(attn_out))

      return attn_out + mlp_out





In [14]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [15]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (7.6ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (1.1ms)
  ✅ [3/5] MLP has 4x expansion with GELU (1.0ms)
  ✅ [4/5] Causal masking — future doesn't affect past (4.0ms)
  ✅ [5/5] Gradient flow to all parameters (4.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (18.1ms total)
  Progress saved. Run status() to see your dashboard.

